# Part 5: Updating Data and Transactions

In this notebook we practice:

- `INSERT` — add new rows
- `UPDATE` — modify existing rows
- `DELETE` — remove rows
- **Transactions** — group multiple statements so they all succeed or all fail together
- `ROLLBACK` — undo changes when something goes wrong

> **Prerequisite:** Run `01_setup.ipynb` first.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("gene_expression.db")
con.execute("PRAGMA foreign_keys = ON")

def query(sql, params=()):
    return pd.read_sql(sql, con, params=params)

print("Connected.")

---
## 1. INSERT — add a new gene

In [ ]:
# Remove the gene if it already exists (for re-runnability)
con.execute("DELETE FROM genes WHERE symbol = 'CCND1'")
con.commit()

# Insert a new gene
con.execute("""
    INSERT INTO genes (symbol, chromosome, strand, start_pos, end_pos, description)
    VALUES ('CCND1', '11', '+', 69641156, 69654473, 'Cyclin D1')
""")
con.commit()

# Verify
query("SELECT * FROM genes WHERE symbol = 'CCND1'")

---
## 2. INSERT OR REPLACE — upsert pattern

If a row violates a UNIQUE constraint, `INSERT OR REPLACE` deletes the old row and inserts the new one.

In [ ]:
con.execute("""
    INSERT OR REPLACE INTO genes (symbol, chromosome, strand, start_pos, end_pos, description)
    VALUES ('CCND1', '11', '+', 69641156, 69654473, 'Cyclin D1 — updated description')
""")
con.commit()

query("SELECT * FROM genes WHERE symbol = 'CCND1'")

---
## 3. UPDATE — modify existing rows

In [ ]:
# Check current description
query("SELECT symbol, description FROM genes WHERE symbol = 'TP53'")

In [ ]:
con.execute("""
    UPDATE genes
    SET    description = 'Tumor protein p53 — guardian of the genome'
    WHERE  symbol = 'TP53'
""")
con.commit()

query("SELECT symbol, description FROM genes WHERE symbol = 'TP53'")

---
## 4. DELETE — remove rows

In [ ]:
# Count before
print("Before:", con.execute("SELECT COUNT(*) FROM genes").fetchone()[0], "genes")

# Delete the gene we just added
con.execute("DELETE FROM genes WHERE symbol = 'CCND1'")
con.commit()

print("After: ", con.execute("SELECT COUNT(*) FROM genes").fetchone()[0], "genes")

---
## 5. Transactions — all-or-nothing updates

A **transaction** groups multiple statements together. Either all of them succeed (`COMMIT`), or none of them take effect (`ROLLBACK`).

This is critical for data integrity — for example, adding a new sample and its expression values should either both succeed or both fail.

In [ ]:
# Successful transaction: add a new sample and expression values

try:
    con.execute("BEGIN")

    # Step 1: insert new sample
    con.execute("""
        INSERT OR IGNORE INTO samples (name, condition, tissue)
        VALUES ('COLON_T01', 'tumor', 'colon')
    """)

    # Step 2: get the new sample_id
    (new_sample_id,) = con.execute(
        "SELECT sample_id FROM samples WHERE name = 'COLON_T01'"
    ).fetchone()

    # Step 3: insert expression rows for all existing genes
    import random
    random.seed(7)
    gene_ids = [r[0] for r in con.execute("SELECT gene_id FROM genes")]
    expr_rows = [
        (gid, new_sample_id, round(random.uniform(10, 3000), 2), random.randint(10, 2000))
        for gid in gene_ids
    ]
    con.executemany(
        "INSERT OR IGNORE INTO expression (gene_id, sample_id, tpm, raw_counts) VALUES (?,?,?,?)",
        expr_rows
    )

    con.execute("COMMIT")
    print(f"Transaction committed. Added sample COLON_T01 with {len(expr_rows)} expression rows.")

except Exception as e:
    con.execute("ROLLBACK")
    print(f"Transaction rolled back due to error: {e}")

In [ ]:
# Verify the new sample exists
query("SELECT * FROM samples WHERE name = 'COLON_T01'")

---
## 6. Rollback — simulating a failure mid-transaction

In [ ]:
print("Samples before:", con.execute("SELECT COUNT(*) FROM samples").fetchone()[0])

try:
    con.execute("BEGIN")

    # This insert will succeed
    con.execute("INSERT INTO samples (name, condition, tissue) VALUES ('LIVER_T01', 'tumor', 'liver')")
    print("Inserted LIVER_T01")

    # Simulate an error (duplicate sample name)
    con.execute("INSERT INTO samples (name, condition, tissue) VALUES ('LIVER_T01', 'tumor', 'liver')")
    print("This line will not be reached")

    con.execute("COMMIT")

except sqlite3.IntegrityError as e:
    con.execute("ROLLBACK")
    print(f"Error caught, rolling back: {e}")

print("Samples after: ", con.execute("SELECT COUNT(*) FROM samples").fetchone()[0])

The count is unchanged — the ROLLBACK undid the first `INSERT` even though it succeeded, because the second one failed.

---
## Exercises

**Q1.** Add a new sample `'BRAIN_N01'` (normal, brain tissue) and insert random expression values for all existing genes. Wrap both steps in a transaction.

In [ ]:
# Your code here


**Q2.** Update all expression `tpm` values for sample `'LUNG_T01'` by multiplying them by 1.1 (a calibration correction). Verify the change.

In [ ]:
# Your code here


**Q3.** Write a transaction that deletes the sample `'COLON_T01'` and all its associated expression rows. What happens if you try to delete only the sample without deleting expression rows first (with foreign keys enabled)?

In [ ]:
# Your code here


In [ ]:
con.close()